# EEG Preprocessing Visualisations — P01 to P10

**Dataset:** `E:/REU_DATA/` — participants **P01–P10**, 4 conditions each (calm, meditation, maths, speech)
**Pipeline notebook:** `EEG_Dataset_Analysis_and_Preprocessing.ipynb`

This notebook is **visualisation only**. Nothing is re-filtered and no raw file is modified. For every
recording it loads

```text
raw .acq      →  BEFORE filtering
cleaned .fif  →  AFTER  1–45 Hz band-pass + 50 Hz notch   (written by the pipeline notebook)
```

and plots it with **the same graph types already used in the preprocessing notebook**:

| # | Graph | Same as |
|---|---|---|
| 1 | Filter frequency response (band-pass + notch) | STEP 10 |
| 2 | Time-domain overlay (raw grey vs cleaned blue) | STEP 12.1 |
| 3 | Multi-channel grid, before / after | STEP 12.2 |
| 4 | PSD comparison (1–80 Hz + 40–60 Hz notch zoom) | STEP 12.3 |
| 5 | STFT spectrogram before / after | STEP 12.4 |
| 6 | Per-channel standard-deviation bar chart | STEP 12.6 |
| 7 | Cross-participant summary bar charts | derived from STEP 12.6 / 14 |

Everything else from the original notebook (inventory tables, file-size profiling, `.xlsx` header
inspection, bad-channel tables, ICA notes) is **deliberately not repeated** — that analysis already
lives in the preprocessing notebook.

**Scope:** all 10 participants × all 4 conditions = **40 recordings** (5 figures each).
Change `PARTICIPANTS_TO_PLOT` / `CONDITIONS_TO_PLOT` in STEP 2 to plot a subset.

**Safety rule (unchanged):** raw `.acq` / `.xlsx` files under `P01`–`P10` are **read-only**. This
notebook only *reads* them; every figure is written under `REU_DATA/processed/eeg/figures/`.

## STEP 1 — Import libraries

- **numpy / pandas** — arrays and summary tables
- **matplotlib** — all diagnostic plots (identical styling to the preprocessing notebook)
- **mne** — reading cleaned `.fif`, PSD, topography-ready info
- **scipy** — filter frequency response + STFT spectrograms
- **bioread** — reading the native BIOPAC `.acq` raw recordings

In [ ]:
import gc
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
from mne import create_info
from mne.io import RawArray
from scipy import signal as sps
from scipy.signal import stft
import bioread

mne.set_log_level("ERROR")  # keep notebook output readable

# compact inline rendering (every figure is also saved at full resolution by _finish())
plt.rcParams["figure.dpi"] = 80
plt.rcParams["savefig.dpi"] = 120

print("Python :", sys.version.split()[0])
print("numpy  :", np.__version__)
print("pandas :", pd.__version__)
print("mne    :", mne.__version__)

## STEP 2 — Paths, participants and what to plot

- `RAW_DATA_DIR` — **read-only** source of truth (`E:/REU_DATA`)
- `PROCESSED_DATA_DIR` — cleaned `.fif` + reports written by the pipeline notebook
- `FIG_DIR` — where this notebook saves every figure

Then choose the scope of the visualisation run.

In [ ]:
RAW_DATA_DIR = Path(r"E:/REU_DATA")
PROCESSED_DATA_DIR = RAW_DATA_DIR / "processed" / "eeg"
FIG_DIR = PROCESSED_DATA_DIR / "figures"

assert RAW_DATA_DIR.exists(), f"Raw data dir not found: {RAW_DATA_DIR}"
FIG_DIR.mkdir(parents=True, exist_ok=True)

CONDITIONS = {
    "01calm": "calm",
    "02meditation": "meditation",
    "03maths": "maths",
    "04speech": "speech",
}
SUFFIX_OF = {v: k for k, v in CONDITIONS.items()}

# --- scope: every participant, every condition ---
PARTICIPANTS_TO_PLOT = [f"P{i:02d}" for i in range(1, 11)]
CONDITIONS_TO_PLOT = ["calm", "meditation", "maths", "speech"]

# representative channels used in the time-domain plots (same as the preprocessing notebook)
PLOT_CHANNELS = ["Fp1", "Fp2", "C3", "Cz", "O1", "O2"]

participant_dirs = {}
for i in range(1, 11):
    pname = f"P{i:02d}"
    acq_dir = RAW_DATA_DIR / pname / f"{i:02d}_eeg" / "acq"
    if acq_dir.exists():
        participant_dirs[pname] = acq_dir

missing = [p for p in PARTICIPANTS_TO_PLOT if p not in participant_dirs]
print(f"Participants available : {len(participant_dirs)}/10" + (f"   missing {missing}" if missing else ""))
print(f"Conditions to plot     : {CONDITIONS_TO_PLOT}")
print(f"Recordings in scope    : {len(PARTICIPANTS_TO_PLOT) * len(CONDITIONS_TO_PLOT)}")
print(f"Figures saved under    : {FIG_DIR}")

## STEP 3 — Verify the filter designs (frequency response)

The filters themselves were designed in the preprocessing notebook; here we re-plot their frequency
response **once**, because it applies identically to all 40 recordings — confirming the 1–45 Hz
band-pass and the 50 Hz notch do what their names claim.

In [ ]:
report_files = sorted(PROCESSED_DATA_DIR.glob("P*/**/preprocessing_report.json"))
FS = 1000
if report_files:
    with open(report_files[0], encoding="utf-8") as f:
        FS = int(json.load(f).get("Sampling rate (Hz)", 1000))
print(f"Using Fs = {FS} Hz")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

fir_hz = sps.firwin(numtaps=FS + 1, cutoff=[1.0, 45.0], fs=FS, window="hamming", pass_zero=False)
freqs, resp = sps.freqz(fir_hz, worN=8192, fs=FS)
axes[0].plot(freqs, 20 * np.log10(np.abs(resp) + 1e-12))
axes[0].set_title("Band-pass 1-45 Hz - frequency response")
axes[0].set_xlabel("Frequency (Hz)")
axes[0].set_ylabel("Gain (dB)")
axes[0].set_ylim(-80, 5)
axes[0].axvline(1, color="r", ls="--", alpha=0.5)
axes[0].axvline(45, color="r", ls="--", alpha=0.5)
axes[0].grid(True, alpha=0.3)

b_notch, a_notch = sps.iirnotch(w0=50.0, Q=30.0, fs=FS)
freqs_n, resp_n = sps.freqz(b_notch, a_notch, worN=8192, fs=FS)
axes[1].plot(freqs_n, 20 * np.log10(np.abs(resp_n) + 1e-12))
axes[1].set_title("50 Hz notch (Q=30) - frequency response")
axes[1].set_xlabel("Frequency (Hz)")
axes[1].set_ylabel("Gain (dB)")
axes[1].set_ylim(-40, 5)
axes[1].axvline(50, color="r", ls="--", alpha=0.5)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(FIG_DIR / "00_filter_frequency_response.png", dpi=120, bbox_inches="tight")
plt.show()

## STEP 4 — Loading and plotting helpers

One loader for raw `.acq` (before) and cleaned `.fif` (after), plus one function per graph type.
Each function saves its figure under `processed/eeg/figures/<participant>/` and shows it inline.

Band definitions (delta/theta/alpha/beta/gamma) match the preprocessing notebook exactly.

In [ ]:
STANDARD_1020 = mne.channels.make_standard_montage("standard_1020")

BANDS = {
    "Delta (1-4 Hz)": (1, 4),
    "Theta (4-8 Hz)": (4, 8),
    "Alpha (8-13 Hz)": (8, 13),
    "Beta (13-30 Hz)": (13, 30),
    "Gamma (30-45 Hz)": (30, 45),
}


def _finish(fig, name):
    """Save the figure under FIG_DIR (subfolder per participant) and display it."""
    out = FIG_DIR / name
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out, dpi=120, bbox_inches="tight")
    plt.show()
    return out


def load_acq_to_raw(acq_path):
    """Read-only: BIOPAC .acq -> MNE RawArray (microvolts -> volts)."""
    df = bioread.read_file(str(acq_path))
    fs = int(df.samples_per_second)
    ch_names = [ch.name for ch in df.channels]
    data = np.vstack([ch.data for ch in df.channels]) * 1e-6
    info = create_info(ch_names=ch_names, sfreq=fs, ch_types="eeg")
    raw = RawArray(data, info, verbose=False)
    raw.set_montage(STANDARD_1020, on_missing="ignore", verbose=False)
    del df, data
    gc.collect()
    return raw


def load_pair(pid, cond):
    """Return (raw_before, raw_after) for one recording, or (None, None) if unavailable."""
    acq_dir = participant_dirs.get(pid)
    acq_matches = list(acq_dir.glob(f"{pid}_*{SUFFIX_OF[cond]}.acq")) if acq_dir else []
    fif_path = PROCESSED_DATA_DIR / pid / cond / f"{pid}_{cond}_eeg.fif"
    if not acq_matches:
        print(f"  SKIP {pid} {cond}: raw .acq not found")
        return None, None
    if not fif_path.exists():
        print(f"  SKIP {pid} {cond}: cleaned .fif not found - run the preprocessing notebook first")
        return None, None
    return load_acq_to_raw(acq_matches[0]), mne.io.read_raw_fif(fif_path, verbose=False)


def _common_channels(raw_before, raw_after, wanted):
    return [c for c in wanted if c in raw_before.ch_names and c in raw_after.ch_names]


def plot_time_overlay(raw_before, raw_after, pid, cond):
    """Graph 2 - raw (gray) vs cleaned (blue) on the same axes."""
    fs = int(raw_before.info["sfreq"])
    seg = min(30, int(raw_before.n_times // fs))
    n = seg * fs
    t = np.arange(n) / fs
    chs = _common_channels(raw_before, raw_after, PLOT_CHANNELS)
    data_b = raw_before.get_data()[:, :n] * 1e6
    data_a = raw_after.get_data()[:, :n] * 1e6

    fig, axes = plt.subplots(len(chs), 1, figsize=(14, 9), sharex=True)
    for ax, ch in zip(np.atleast_1d(axes), chs):
        i, j = raw_before.ch_names.index(ch), raw_after.ch_names.index(ch)
        ax.plot(t, data_b[i], lw=0.4, color="gray", alpha=0.7, label="Raw")
        ax.plot(t, data_a[j], lw=0.4, color="C0", alpha=0.9, label="Cleaned")
        ax.set_ylabel(f"{ch}\n(µV)")
        ax.grid(True, alpha=0.2)
    np.atleast_1d(axes)[0].legend(loc="upper right", fontsize=9)
    np.atleast_1d(axes)[0].set_title(
        f"Time-domain: raw (gray) vs cleaned (blue) - {pid} {cond} - {seg}s", fontsize=13)
    np.atleast_1d(axes)[-1].set_xlabel("Time (s)")
    plt.tight_layout()
    _finish(fig, f"{pid}/{pid}_{cond}_1_time-domain.png")


def plot_channel_grid(raw_before, raw_after, pid, cond):
    """Graph 3 - 3x2 multi-channel grid, BEFORE figure then AFTER figure."""
    fs = int(raw_before.info["sfreq"])
    seg = min(30, int(raw_before.n_times // fs))
    n = seg * fs
    chs = _common_channels(raw_before, raw_after, PLOT_CHANNELS)

    data_b = raw_before.get_data()[:, :n] * 1e6
    fig, axes = plt.subplots(3, 2, figsize=(15, 9), sharex=True)
    for ax, ch in zip(axes.flat, chs):
        ax.plot(np.arange(n) / fs, data_b[raw_before.ch_names.index(ch)], lw=0.4, color="gray")
        ax.set_title(f"{ch} - BEFORE", fontsize=10)
        ax.set_ylabel("µV")
        ax.grid(True, alpha=0.2)
    for ax in axes[-1]:
        ax.set_xlabel("Time (s)")
    fig.suptitle(f"BEFORE preprocessing - raw EEG - {pid} {cond} (DC offsets & 50 Hz fuzz)", fontsize=13)
    plt.tight_layout()
    _finish(fig, f"{pid}/{pid}_{cond}_2_grid-before.png")

    data_a = raw_after.get_data()[:, :n] * 1e6
    fig, axes = plt.subplots(3, 2, figsize=(15, 9), sharex=True)
    for ax, ch in zip(axes.flat, chs):
        ax.plot(np.arange(n) / fs, data_a[raw_after.ch_names.index(ch)], lw=0.4, color="C0")
        ax.set_title(f"{ch} - AFTER", fontsize=10)
        ax.set_ylabel("µV")
        ax.grid(True, alpha=0.2)
    for ax in axes[-1]:
        ax.set_xlabel("Time (s)")
    fig.suptitle(f"AFTER preprocessing - 1-45 Hz band-pass + 50 Hz notch - {pid} {cond}", fontsize=13)
    plt.tight_layout()
    _finish(fig, f"{pid}/{pid}_{cond}_3_grid-after.png")


def plot_psd_comparison(raw_before, raw_after, pid, cond):
    """Graph 4 - mean PSD over channels: full 1-80 Hz + zoom on the 50 Hz notch."""
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    for raw_, label, color in [(raw_before, "Raw", "gray"), (raw_after, "Cleaned", "C0")]:
        psds, freqs = raw_.compute_psd(fmin=1, fmax=80, verbose=False).get_data(return_freqs=True)
        axes[0].plot(freqs, 10 * np.log10(psds.mean(axis=0) + 1e-20), label=label, color=color, lw=1.2)
    axes[0].axvline(50, color="r", ls="--", alpha=0.4)
    axes[0].set_xlabel("Frequency (Hz)")
    axes[0].set_ylabel("PSD (dB)")
    axes[0].set_title("PSD before vs after (1-80 Hz)")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    for raw_, label, color in [(raw_before, "Raw", "gray"), (raw_after, "Cleaned", "C0")]:
        psds, freqs = raw_.compute_psd(fmin=40, fmax=60, verbose=False).get_data(return_freqs=True)
        axes[1].plot(freqs, 10 * np.log10(psds.mean(axis=0) + 1e-20), label=label, color=color, lw=1.2)
    axes[1].axvline(50, color="r", ls="--", alpha=0.4, label="50 Hz")
    axes[1].set_xlabel("Frequency (Hz)")
    axes[1].set_ylabel("PSD (dB)")
    axes[1].set_title("Zoom: 40-60 Hz (notch effect)")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.suptitle(f"PSD comparison - {pid} {cond} (mean over {len(raw_before.ch_names)} channels)", fontsize=13, y=1.02)
    plt.tight_layout()
    _finish(fig, f"{pid}/{pid}_{cond}_4_psd.png")


def plot_spectrogram(raw_before, raw_after, pid, cond):
    """Graph 5 - STFT spectrogram before vs after for a representative channel."""
    ch = "Cz" if "Cz" in raw_before.ch_names else raw_before.ch_names[0]
    fs = int(raw_before.info["sfreq"])
    x_before = raw_before.get_data(picks=ch)[0] * 1e6
    x_after = raw_after.get_data(picks=ch)[0] * 1e6

    nperseg = 2048  # ~2 s window
    f_b, t_b, Zxx_b = stft(x_before, fs=fs, nperseg=nperseg, noverlap=nperseg // 2)
    f_a, t_a, Zxx_a = stft(x_after, fs=fs, nperseg=nperseg, noverlap=nperseg // 2)

    fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
    im0 = axes[0].pcolormesh(t_b, f_b, 10 * np.log10(np.abs(Zxx_b) + 1e-12), shading="gouraud", cmap="magma")
    axes[0].set_ylabel("Frequency (Hz)")
    axes[0].set_title(f"Spectrogram BEFORE - {ch} ({pid} {cond}: line noise & low-freq drift)")
    axes[0].set_ylim(0, 80)
    plt.colorbar(im0, ax=axes[0], label="dB")

    im1 = axes[1].pcolormesh(t_a, f_a, 10 * np.log10(np.abs(Zxx_a) + 1e-12), shading="gouraud", cmap="magma")
    axes[1].set_ylabel("Frequency (Hz)")
    axes[1].set_xlabel("Time (s)")
    axes[1].set_title(f"Spectrogram AFTER - {ch} (50 Hz removed, 1-45 Hz band isolated)")
    axes[1].set_ylim(0, 80)
    plt.colorbar(im1, ax=axes[1], label="dB")

    plt.tight_layout()
    _finish(fig, f"{pid}/{pid}_{cond}_5_spectrogram.png")


def plot_std_bars(raw_before, raw_after, pid, cond):
    """Graph 6 - per-channel standard deviation before vs after. Returns the numbers."""
    stds_before = raw_before.get_data().std(axis=1) * 1e6
    stds_after = raw_after.get_data().std(axis=1) * 1e6

    x = np.arange(len(raw_before.ch_names))
    width = 0.38

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(x - width / 2, stds_before, width, label="Raw", color="gray", alpha=0.8)
    ax.bar(x + width / 2, stds_after, width, label="Cleaned", color="C0", alpha=0.9)
    ax.set_xticks(x)
    ax.set_xticklabels(raw_before.ch_names, rotation=45, ha="right")
    ax.set_ylabel("Std (µV)")
    ax.set_title(f"Per-channel standard deviation - before vs after ({pid} {cond})")
    ax.legend()
    ax.grid(True, axis="y", alpha=0.3)

    reduction = (1 - stds_after.mean() / stds_before.mean()) * 100
    ax.text(0.98, 0.95, f"Mean std reduction: {reduction:.1f}%",
            transform=ax.transAxes, ha="right", va="top",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))
    plt.tight_layout()
    _finish(fig, f"{pid}/{pid}_{cond}_6_std-bars.png")

    return {
        "Mean std raw (µV)": round(float(stds_before.mean()), 2),
        "Mean std cleaned (µV)": round(float(stds_after.mean()), 2),
        "Std reduction (%)": round(float(reduction), 1),
    }


def band_powers(raw):
    """Mean band power (dB) over channels for delta/theta/alpha/beta/gamma."""
    psds, freqs = raw.compute_psd(fmin=1, fmax=45, verbose=False).get_data(return_freqs=True)
    out = {}
    for name, (fmin, fmax) in BANDS.items():
        mask = (freqs >= fmin) & (freqs <= fmax)
        out[name] = round(float(10 * np.log10(psds[:, mask].mean() + 1e-20)), 2)
    return out


print(f"Helpers ready: {len(BANDS)} bands, {len(PLOT_CHANNELS)} representative channels")

## STEP 5 — Before/after visualisations for every recording (P01–P10 × 4 conditions)

For each recording in scope we plot the five core graph types:

1. **Time-domain overlay** — raw (gray) vs cleaned (blue) on the same axes, 30 s segment
2. **Multi-channel grid** — 6 representative channels, BEFORE figure and AFTER figure
3. **PSD comparison** — mean PSD 1–80 Hz + 40–60 Hz zoom showing the notch effect
4. **Spectrogram (STFT)** — time-frequency view before vs after (Cz)
5. **Channel variance bar chart** — per-channel standard deviation before vs after (with % reduction)

Band power (delta/theta/alpha/beta/gamma) is computed from the cleaned data in the same pass and kept
for the cross-participant summary in STEP 6. Memory is freed after every recording, and one failing
recording never aborts the run.

In [ ]:
records = []

for pid in PARTICIPANTS_TO_PLOT:
    print(f"\n=== {pid} ===")
    for cond in CONDITIONS_TO_PLOT:
        try:
            raw_before, raw_after = load_pair(pid, cond)
            if raw_before is None:
                continue

            plot_time_overlay(raw_before, raw_after, pid, cond)
            plot_channel_grid(raw_before, raw_after, pid, cond)
            plot_psd_comparison(raw_before, raw_after, pid, cond)
            plot_spectrogram(raw_before, raw_after, pid, cond)

            row = {
                "Participant": pid,
                "Condition": cond,
                "Duration (s)": round(raw_before.n_times / raw_before.info["sfreq"], 1),
                "Channels": len(raw_before.ch_names),
            }
            row.update(plot_std_bars(raw_before, raw_after, pid, cond))
            row.update(band_powers(raw_after))

            report_path = PROCESSED_DATA_DIR / pid / cond / "preprocessing_report.json"
            if report_path.exists():
                with open(report_path, encoding="utf-8") as f:
                    rep = json.load(f)
                row["Notch"] = rep.get("Notch", "—")
                excluded = rep.get("Channels excluded", [])
                row["Bad chans (interp)"] = ", ".join(excluded) if isinstance(excluded, list) and excluded else "—"

            records.append(row)
            print(f"  OK   {cond:<12} std reduction {row.get('Std reduction (%)', float('nan'))}%")

            del raw_before, raw_after
            gc.collect()
        except Exception as e:
            print(f"  FAIL {cond}: {e}")
            gc.collect()

vis_summary = pd.DataFrame(records)
print(f"\nVisualised {len(vis_summary)} / {len(PARTICIPANTS_TO_PLOT) * len(CONDITIONS_TO_PLOT)} recordings.")

**What was done:**
• Loaded raw `.acq` (before) and cleaned `.fif` (after) for every recording in scope and produced the
  five core graphs for each, saving them under `processed/eeg/figures/<participant>/`.

**Why:**
• The same graph types as the preprocessing notebook, but now applied uniformly across P01–P10 — this
  is what proves the pipeline behaved identically for every participant, not just for the P01 pilot.

**Result:**
• Per-recording visual proof plus one metrics row per recording (`vis_summary`), used next.

## STEP 6 — Cross-participant summary (required analysis only)

Three compact aggregates — no per-recording graphs repeated here:

1. A metrics table (duration, std reduction, band power, notch, interpolated channels) saved as CSV
2. **Bar chart:** mean std reduction per participant, grouped by condition
3. **Grouped bar chart:** mean band power per condition, averaged over P01–P10

In [ ]:
if len(vis_summary) == 0:
    raise RuntimeError("No recordings were visualised - check that cleaned .fif files exist.")

display(vis_summary)

summary_csv = PROCESSED_DATA_DIR / "preprocessing_visualisation_summary.csv"
vis_summary.to_csv(summary_csv, index=False)
print(f"Saved: {summary_csv}")

In [ ]:
# ---- Bar chart: mean std reduction (%) per participant, grouped by condition ----
piv = vis_summary.pivot_table(index="Participant", columns="Condition",
                               values="Std reduction (%)")
piv = piv.reindex(index=[p for p in PARTICIPANTS_TO_PLOT if p in set(vis_summary["Participant"])])
conds = [c for c in CONDITIONS_TO_PLOT if c in piv.columns]

x = np.arange(len(piv.index))
width = 0.2
colors = ["C0", "C1", "C2", "C3"]

fig, ax = plt.subplots(figsize=(13, 5))
for i, cond in enumerate(conds):
    ax.bar(x + i * width - 1.5 * width, piv[cond].to_numpy(), width, label=cond,
           color=colors[i], alpha=0.9)
ax.set_xticks(x)
ax.set_xticklabels(piv.index)
ax.set_ylabel("Mean std reduction (%)")
ax.set_title("Effect of preprocessing on signal variance - per participant (grouped by condition)")
ax.legend(title="Condition")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Grouped bar chart: band power by condition, averaged over P01–P10 ----
band_cols = list(BANDS.keys())
band_df = pd.DataFrame([
    {"Condition": r["Condition"], **{b: r.get(b) for b in band_cols}} for r in records
])
band_summary = (band_df.groupby("Condition")[band_cols]
                       .mean()
                       .reindex([c for c in CONDITIONS_TO_PLOT if c in set(band_df["Condition"])]))

display(band_summary.round(2))

x = np.arange(len(band_cols))
width = 0.2
colors = ["C0", "C1", "C2", "C3"]

fig, ax = plt.subplots(figsize=(13, 6))
for i, cond in enumerate(band_summary.index):
    ax.bar(x + i * width - 1.5 * width, band_summary.loc[cond].to_numpy(), width,
           label=cond, color=colors[i], alpha=0.9)
ax.set_xticks(x)
ax.set_xticklabels(band_cols, rotation=15)
ax.set_ylabel("Mean band power (dB)")
ax.set_title("Band power by condition - mean over participants (cleaned data)")
ax.legend(title="Condition")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

```text
raw .acq  (P01–P10 × calm/meditation/maths/speech)
        ↓  read-only
BEFORE / AFTER pairs per recording
        ↓
5 core graphs per recording (same types as the preprocessing notebook):
  time-domain overlay · multi-channel grid · PSD · spectrogram · std bar chart
        ↓
cross-participant aggregates:
  metrics CSV · std-reduction bar chart · band-power grouped bar chart
```

- **Raw data was never modified** — `.acq` / `.xlsx` files under `P01`–`P10` are untouched.
- Figures are saved in `REU_DATA/processed/eeg/figures/<participant>/`.
- Metrics table: `REU_DATA/processed/eeg/preprocessing_visualisation_summary.csv`.
- To plot a subset, edit `PARTICIPANTS_TO_PLOT` / `CONDITIONS_TO_PLOT` in STEP 2 and re-run.